## Lab 4 - Cross-encoder re-ranking

In [6]:
!pip install langchain-text-splitters
!pip install chromadb
!pip install sentence_transformers
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter

def load_chroma(filename, collection_name, embedding_function):

    # Read PDF
    reader = PdfReader(filename)
    pdf_texts = [page.extract_text().strip() for page in reader.pages]

    # Remove empty pages
    pdf_texts = [text for text in pdf_texts if text]

    # Character splitting
    character_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=1000,
        chunk_overlap=0
    )

    character_split_texts = character_splitter.split_text(
        "\n\n".join(pdf_texts)
    )

    # Token splitting
    token_splitter = SentenceTransformersTokenTextSplitter(
        chunk_overlap=0,
        tokens_per_chunk=256
    )

    token_split_texts = []

    for text in character_split_texts:
        token_split_texts.extend(
            token_splitter.split_text(text)
        )

    # Create Chroma collection
    chroma_client = chromadb.Client()

    try:
        chroma_client.delete_collection(collection_name)
        print(f"Deleted existing collection: {collection_name}")
    except:
        pass

    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=embedding_function
    )

    # Add documents
    ids = [str(i) for i in range(len(token_split_texts))]

    collection.add(
        ids=ids,
        documents=token_split_texts
    )

    return collection


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [7]:

import textwrap

def word_wrap(text, width=80):
    return textwrap.fill(text, width=width)

import numpy as np
from tqdm import tqdm

def project_embeddings(embeddings, umap_transform):
    """
    Project high-dimensional embeddings into 2D using a trained UMAP model.
    """
    umap_embeddings = np.empty((len(embeddings), 2))

    for i, embedding in enumerate(tqdm(embeddings)):
        umap_embeddings[i] = umap_transform.transform([embedding])

    return umap_embeddings

In [8]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import numpy as np

In [12]:
!pip install pypdf
from pypdf import PdfReader

embedding_function = SentenceTransformerEmbeddingFunction()

chroma_collection = load_chroma(filename='sample_data/2025_AnnualReport.pdf', collection_name='2025_AnnualReport', embedding_function=embedding_function)
chroma_collection.count()

ERROR: Operation cancelled by user


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

315

# Re-ranking the long tail

In [13]:
query = "What has been the investment in research and development?"
results = chroma_collection.query(query_texts=query, n_results=10, include=['documents', 'embeddings'])

retrieved_documents = results['documents'][0]

for document in results['documents'][0]:
    print(word_wrap(document))
    print('')

operating costs related to product support service centers and product
distribution centers ; traffic acquisition costs to drive traffic to our
websites and to acquire online advertising space ; and costs associated with the
delivery of consulting services. research and development research and
development expenses include payroll, employee benefits, stock - based
compensation expense, and other headcount - related expenses associated with
product development. research and development expenses also include third -
party development and programming costs and the depreciation and amortization of
assets used to conduct research and development. such costs related to software
development are included in research and development expense until the point
that technological feasibility is reached, which for our software products is
generally shortly before the products are released to production. once
technological feasibility is reached, such costs are capitalized and

note 4 — investments in

In [14]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [15]:
pairs = [[query, doc] for doc in retrieved_documents]
scores = cross_encoder.predict(pairs)
print("Scores:")
for score in scores:
    print(score)

Scores:
-3.1892045
-6.4312744
-5.62525
0.57847977
5.313188
-0.9027629
-5.852653
-10.575176
-10.942207
-5.5652924


In [16]:
print("New Ordering:")
for o in np.argsort(scores)[::-1]:
    print(o+1)

New Ordering:
5
4
6
1
10
3
7
2
8
9


# Re-ranking with Query Expansion

In [17]:
original_query = "What were the most important factors that contributed to increases in revenue?"
generated_queries = [
    "What were the major drivers of revenue growth?",
    "Were there any new product launches that contributed to the increase in revenue?",
    "Did any changes in pricing or promotions impact the revenue growth?",
    "What were the key market trends that facilitated the increase in revenue?",
    "Did any acquisitions or partnerships contribute to the revenue growth?"
]

In [18]:
queries = [original_query] + generated_queries

results = chroma_collection.query(query_texts=queries, n_results=10, include=['documents', 'embeddings'])
retrieved_documents = results['documents']

In [19]:
# Deduplicate the retrieved documents
unique_documents = set()
for documents in retrieved_documents:
    for document in documents:
        unique_documents.add(document)

unique_documents = list(unique_documents)

In [20]:
pairs = []
for doc in unique_documents:
    pairs.append([original_query, doc])

In [21]:
scores = cross_encoder.predict(pairs)


In [22]:
print("Scores:")
for score in scores:
    print(score)

Scores:
-10.051771
-5.039777
-10.177038
-9.62886
-4.924861
-10.957497
-11.143802
-10.129626
-10.344662
-9.292367
-10.287441
-7.0859985
-10.10235
-8.759403
-2.6663983
-11.005256
-8.124691
-10.773321
-10.426801
-5.195134
-8.211106
-4.5752573
-6.622036
-7.4822044
-10.113169


In [23]:
print("New Ordering:")
for o in np.argsort(scores)[::-1]:
    print(o)

New Ordering:
14
21
4
1
19
22
11
23
16
20
13
9
3
0
12
24
7
2
10
8
18
17
5
15
6
